1250 grid points in cosmology 

Each grid point generate ~150 square maps from one full sky realization, to sum up the total area to DES survey

In each square map we have 50 by 50 pixels, with map resolution equal to that from NSIDE 512

The data set at one cosmology is of size (150, 1, 50, 50) which would return 150 compressed data vectors of $(c_1, c_2)$ after inputing into C3NN. We then need an extra layer of operation to average these 150 output to $(\bar{c}_1, \bar{c}_2)$. Then $\{(\bar{c}_1, \bar{c}_2), \theta\}$ forms one argument in the SBI framework where $\theta$ is one cosmological parameter vector. In total we would have 1250 of them.

1. How to split training and testing data set on the dimension of cosmology in SBI
2. Modify the C3NN part such that it can average the 150 output {c} vectors
3. Find a solution to the detach and manual deepcopy of the trained model such that we can continue to train a model after saving it

In [1]:
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import equivariant steerable cnn
import escnn.gspaces
import escnn.nn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import from_numpy
from sklearn.model_selection import train_test_split

from tqdm import tqdm
# Checking if you have a GPU and CUDA set up -- if not, training will take longer by running on the CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.empty_cache()

from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn

from os import listdir
from os.path import isfile, join

from copy import deepcopy

#from getdist import MCSamples, plots

import pickle as pk

from hyperparameters import hparams

In [25]:
training_data = np.load('/e/ocean1/users/lgong/CCNN/cosmogrid/kappa_noiseless_1250_grid_cosmologies_150_maps_50_by_50pixels_6_by_6deg_no_smooth_4_des_zsbin_gnom_projection.npy')

In [26]:
training_data = np.reshape(training_data, (-1, 4, 50, 50))

In [27]:
training_data = torch.tensor(training_data, dtype=torch.float32)

In [3]:
training_data.shape

(1250, 150, 4, 50, 50)

In [34]:
training_parameters = np.random.rand(1250, 4)

In [35]:
training_parameters = np.repeat(training_parameters, 150, axis=0)

In [36]:
training_parameters = torch.tensor(training_parameters, dtype=torch.float32)

In [39]:
# set up C3NN
bandlimit = 6
gspace = escnn.gspaces.rot2dOnR2(N=-1, maximum_frequency=bandlimit)


""" Implements the non-linear convolution operation using the recursive formula."""
class NonlinearConv(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order):
        super().__init__()
        self.order = order
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size

        """
        # Initialize filter similar as to how Pytorch initializes nn.Conv2d, but all positive
        # This is the old way of initializing C3NN filters without rotational equivariance/invariance
        weight_sample_k = 1.0 / np.sqrt(filter_size**2 * in_channels)
        init_filt = (2 * weight_sample_k * torch.rand(num_filts, in_channels, filter_size, filter_size, dtype=torch.float32))
        self.conv_filt = torch.nn.Parameter(init_filt)
        # Initialize some slices of the filter to be 0
        with torch.no_grad():
            self.conv_filt[:, :, int(self.filter_size/2)-2:int(self.filter_size/2)+3, int(self.filter_size/2)-2:int(self.filter_size/2)+3]=0.0
        """
        
        # Initialize the escnn filter
        feat_type_in = escnn.nn.FieldType(gspace, self.in_channels*[gspace.trivial_repr])
        feat_type_out = escnn.nn.FieldType(gspace, self.num_filts*[gspace.trivial_repr])
        # THIS LAYER KILLS DEEPCOPY
        self.conv = escnn.nn.R2Conv(in_type=feat_type_in, out_type=feat_type_out, bias=False, kernel_size=self.filter_size) 
        
        # to enable deepcopy
        self.detached = False
        
    def forward(self, x):
        # We will work with strictly positive filters for easier interpretation!
        if self.detached == False:
            self.abs_filt = self.conv.expand_parameters()[0].abs()
        elif self.detached == True:
            pass
        
        # Convolutions of powers of the input with powers of the absolute valued filter
        conv_powers = [F.conv2d(x ** k, self.abs_filt ** k, stride=1, padding=int((self.filter_size-1)/2.0)) for k in range(1, self.order + 1)]
        
        # C_0 is defined to be one everywhere
        corrs = [torch.ones_like(conv_powers[0])]

        # Compute each order of C_n recursively
        for n in range(1, self.order + 1):
            Cn = torch.zeros_like(conv_powers[0])
            for k in range(n):
                Cn += (-1) ** k * conv_powers[k] * corrs[n - k - 1]
            Cn /= n
            corrs.append(Cn)

        # Stack correlator maps along the channel dimension, dropping the zeroth order (C_0)
        # Here the channel dimension refers to the number of channels
        #return torch.cat(corrs[1:], dim=-3)
        return torch.cat(corrs[1:], dim=-3)

    def _detach(self):
        del self.conv
        self.abs_filt = self.abs_filt.detach()
        self.detached = True


""" Does a simple average over the last two dimensions of the input."""
class Mean2d(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return x.mean((-2, -1))


""" The full CCNN is Input -> NonlinearConv -> Spatial Avg -> BatchNorm -> Logistic. The way it is constructed
    here assumes a binary classification task, but this can be extended to the multiclass case."""
class C3NN(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order, absbeta=False, slice=False, pool=False):
        super().__init__()
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size
        self.order = order
        self.absbeta = absbeta

        # the model architecture with escnn
        self.corr_conv = NonlinearConv(self.in_channels, self.num_filts, self.filter_size, self.order)
        self.spatial_mean = Mean2d()
        # I preserve the batch normalization layer here so that we can scale the compressed summary statistics to the same order of magnitude
        # This may help the numerical stability of inference in the 2nd part
        self.bn = nn.BatchNorm1d(num_filts * order, affine=True, eps=3.185289e-35, dtype=torch.float32)


        #troubleshooting
        #self.conv = Conv2d(1,1,11)
    
    def forward(self, x, chop=True):    
        
        x = self.corr_conv(x)
        #x = self.conv(x)
        x = self.spatial_mean(x)
        if chop == False:
            x = self.bn(x)


        return x

    def detach(self):
        self.corr_conv._detach()


#=== 2) Set up some things for training ===#
# The number of input channels 
in_channels = 4
# The number of filters to learn in our model
num_filts = 1
# The spatial length of the filters
filter_size = 35
# The highest order of correlation measured
order = 2
# Make our model -- for simplicity, let's train a two-filter model

In [41]:
c3nn = C3NN(in_channels=in_channels, num_filts=num_filts, filter_size=filter_size, order=order, absbeta=False).to(device='cuda')

# Set up prior for sbi
lower = np.array([0.0, 0.0, 0.0, 0.0])
upper = np.array([1.0, 1.0, 1.0, 1.0])
prior = utils.BoxUniform(low=from_numpy(lower), high=from_numpy(upper),device='cuda')


#set up sbi architecture
density_estimator_custom = posterior_nn(model="maf", hidden_features=hparams.flow_hidden_features, num_transforms=hparams.flow_num_transforms, device='cuda', embedding_net=c3nn)
inference = SNPE_C(prior, device='cuda', density_estimator=density_estimator_custom)

# joint training
density_estimator = inference.append_simulations(training_parameters[:500], training_data[:500]).train(learning_rate=hparams.learning_rate,training_batch_size=hparams.flow_training_batch_size,max_num_epochs=2)
# DANGER DANGER
c3nn.detach()
# now deepcopy works:
test = deepcopy(density_estimator._embedding_net)

/home/moon/lgong/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(
/home/moon/lgong/C3NN_inference/sbi/utils/user_input_checks.py:711: UserWarning: Data x has device 'cpu'.Moving x to the data_device 'cuda:0'.Training will proceed on device 'cuda:0'.
  warnings.warn(
/home/moon/lgong/C3NN_inference/sbi/utils/user_input_checks.py:719: UserWarning: Parameters theta has device 'cpu'. Moving theta to the data_device 'cuda:0'.Training will proceed on device 'cuda:0'.
  warnings.warn(
/home/moon/lgong/C3NN_inference/sbi/utils/user_input_checks.py:435: UserWarning: Mismatch between the device of the data fed to the embedding_net and the device of the embedding_net's

 Training neural network. Epochs trained: 3DANGER!!!! No more deepycopy after training!!!


In [42]:
torch.save(inference, 'trial_inference.pt')

/home/moon/lgong/C3NN_inference/sbi/inference/base.py:429: UserWarning: When the inference object is pickled, the behaviour of the loaded object changes in the following two ways: 1) `.train(..., retrain_from_scratch=True)` is not supported. 2) When the loaded object calls the `.train()` method, it generates a new tensorboard summary writer (instead of appending to the current one).
  warn(


AttributeError: Can't pickle local object '_build_irrep_so2.<locals>.irrep'

In [43]:
torch.save(density_estimator, 'trial_density_estimator.pt')

AttributeError: Can't pickle local object '_build_irrep_so2.<locals>.irrep'